# 02_04 The sliding window

*📖 Book: section 2.6*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → [numbers] → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **numbers**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** Before the model can learn, it needs practice questions with answers. Where do they come from, and how does one piece of text give many of them?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What the model practises on: flashcards made from the text itself
- Why nobody has to write the answers by hand
- How one chunk of text becomes many flashcards at once

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **input** | the text the model is shown: the question on the flashcard | `ἄν` |
| **target** | the character that really comes next: the answer on the flashcard | `δ` |
| **self-supervised learning** | learning from examples whose answers come from the data itself, so no person has to label them | the next character is already in the text |
| **chunk** | a short piece of the text, cut out to practise on; `chunk_size` is its length | 24 characters |

## 3. The idea

### ❓ What does the model actually practise on?

Practice questions with answers, like **flashcards**. For a language model every question is the same kind: "here is some text, what comes next?"

### ❓ Who writes all those flashcards?

Nobody. We make them from the Odyssey itself. Take the first word, `ἄνδρα`:

```
question (input)  →  answer (target)
ἄ                 →  ν
ἄν                →  δ
ἄνδ               →  ρ
ἄνδρ              →  α
```

The answer is always just the next character, and the text already has it. That's why it's called **self-supervised** learning.

### ❓ How many flashcards does one word give?

`ἄνδρα` has 5 characters and gives **4** flashcards. The last character has nothing after it, so it can only ever be an answer, never a question.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How do I make the flashcards in code?

With a loop that cuts the word at every position. The first `n` characters are the question, and the character right after them is the answer.

👉 **Predict first:** how many lines will it print?

In [ ]:
word = "ἄνδρα"
for n in range(1, len(word)):
    print(word[:n], "→", word[n])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

So, 4 lines, one flashcard each, ending with `ἄνδρ → α`.

</details>

### ❓ How does the training code make them all at once?

It doesn't loop over flashcards at all. It takes a **chunk** of the text, plus a second copy shifted one place to the right (`src/homer_ai/train_homer.py`, in `train()`):

```python
inputs  = data[i     : i + chunk_size]
targets = data[i + 1 : i + chunk_size + 1]
```

Read the two copies top to bottom, one column at a time. Each column is one flashcard.

👉 **Predict first:** with a chunk of 4 from the start of Book 1, what are the 4 columns?

In [ ]:
from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

text = load_books(BOOKS_DIR)[0]
tokenizer = CharTokenizer(text)
data = tokenizer.encode(text)

chunk_size = 4
inputs = data[0:chunk_size]
targets = data[1 : chunk_size + 1]

print("inputs: ", tokenizer.decode(inputs))
print("targets:", tokenizer.decode(targets))
for k in range(chunk_size):
    question = tokenizer.decode(inputs[: k + 1])
    answer = tokenizer.decode([targets[k]])
    print(f"{question:>5} → {answer}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`inputs` is `ἄνδρ` and `targets` is `νδρα`: the same text, moved one place. Column by column, they give the same 4 flashcards as the loop did.

</details>

Here are the same two rows as aligned columns, one letter per column, made from the real slices (`chunk_size = 4`, `i = 0`):

In [ ]:
i = 0
inputs = data[i : i + chunk_size]
targets = data[i + 1 : i + chunk_size + 1]

print("inputs :", "  ".join(tokenizer.decode([t]) for t in inputs))
print("targets:", "  ".join(tokenizer.decode([t]) for t in targets))

Read each column top to bottom. Each one is a flashcard, and that shift by one letter is the whole idea.

**Look at:** the top row is the text. The orange chunk is the input and the green one is the target, one place later. Below it, each flashcard is a growing orange window with its green answer.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    GOOD,
    INK,
    MUTED,
    bracket,
    chips,
    diagram_axes,
    studio_figure,
    tag,
)

fig = studio_figure("The sliding window", "Inputs, and the next letter as the answer")
ax = diagram_axes(fig)
phrase = "ἄνδρα μοι"
letters = [c if c != " " else "␣" for c in phrase]
LEFT, STEP = 1.3, 0.8  # where the first chip starts, and the distance between chips

tag(ax, 0.1, 4.6, "the text", MUTED, 9, ha="left")
chips(ax, LEFT, 4.6, letters, 0.7, 0.1)
bracket(ax, LEFT, LEFT + 4 * STEP - 0.1, 4.05, "inputs: a chunk of 4", ACCENT, True)
bracket(ax, LEFT + STEP, LEFT + 5 * STEP - 0.1, 5.15, "targets: one later", GOOD)

tag(ax, 0.1, 3.0, "flashcards", MUTED, 9, ha="left")
for k in range(1, 5):
    y = 3.0 - 0.8 * (k - 1)
    colors = [ACCENT] * k + [GOOD]
    chips(ax, LEFT, y, letters[: k + 1], 0.7, 0.1, colors)
    label = f"{phrase[:k]} → {phrase[k]}"
    tag(ax, LEFT + (k + 1) * STEP + 0.3, y, label, INK, 10, ha="left")
plt.show()

### ❓ Your turn: which flashcards start at `i = 6`?

So far the window sat at the start of the text, `i = 0`. Training slides it along. Move it to `i = 6`, still with `chunk_size = 4`, and cut the inputs and the targets yourself.

Book 1 starts `ἄνδρα μοι ἔννεπε`. Counting from 0, character 6 is the `μ`.

👉 **Predict first:** write down the 4 input characters and the 4 target characters. Careful, one of them is a space.

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
start = 6

# 👉 do: cut the 4 inputs that start at `start`, and the 4 targets one place later
# 💡 hint: the `i = 0` cell did the same; data is the whole text as IDs
window_inputs = None
window_targets = None

In [ ]:
from homer_ai.lessons import check

check(
    "inputs at i = 6",
    window_inputs,
    tokenizer.encode("μοι "),
    show=tokenizer.decode,
    coach="The inputs are 4 IDs: from `start` up to (not including) "
    "`start + chunk_size`.",
)
check(
    "targets at i = 6",
    window_targets,
    tokenizer.encode("οι ἔ"),
    show=tokenizer.decode,
    coach="The targets are the inputs moved one place to the right, so BOTH ends "
    "of the slice get + 1. With only the start moved you get 3 characters, 'οι '.",
)

<details><summary>💡 Hint</summary>

Take the two slices of the `i = 0` cell and write `start` where it has `i`: `data[start : start + chunk_size]`, and the same with `+ 1` on both ends for the targets.

</details>

<details><summary>✅ Answer</summary>

```python
start = 6

window_inputs = data[start : start + chunk_size]
window_targets = data[start + 1 : start + chunk_size + 1]
```

```
inputs : μ  ο  ι  ␣
targets: ο  ι  ␣  ἔ
```

(`␣` stands for the space.) Read the columns: after `μ` comes `ο`, after `μο` comes `ι`, after `μοι` a space, and after `μοι ` the `ἔ` of the next word. So the model also has to learn where words end.

</details>

### ❓ Our `chunk_size` is 24. How many flashcards is that?

👉 **Predict first**, then run.

In [ ]:
chunk_size = 24
inputs = data[0:chunk_size]
targets = data[1 : chunk_size + 1]
print(len(inputs), "inputs,", len(targets), "targets")
print(repr(tokenizer.decode(inputs)))
print(repr(tokenizer.decode(targets)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

**24 flashcards** from one chunk. Each target is the input moved one place to the right. `repr()` shows the text inside quote marks, so you can see that the last target is a space (the character after `μοῦσα,`).

</details>

### ❓ Should the stop be `len(data) - chunk_size`, or one less?

Training walks along the text in steps of `chunk_size`: `for i in range(0, len(data) - chunk_size, chunk_size)`. The targets reach one letter further than the inputs, so it's tempting to think the stop needs an extra `- 1`. Let's check on a text small enough to count. Take 9 letters, `ἄνδρα μοι`, with `chunk_size = 4` (`␣` is the space):

```
position:  0  1  2  3  4  5  6  7  8
letter:    ἄ  ν  δ  ρ  α  ␣  μ  ο  ι
i = 0:     inputs 0-3 (ἄ ν δ ρ)   targets 1-4 (ν δ ρ α)
i = 4:     inputs 4-7 (α ␣ μ ο)   targets 5-8 (␣ μ ο ι)   <- fits: the last target is position 8
```

`range` leaves out its stop value. With the extra `- 1`, the stop `len(data) - chunk_size - 1` would give `i = 0` only, and `␣ μ ο ι` would be wasted. With `len(data) - chunk_size` it gives `i = 0` and `i = 4`.

👉 **Predict first:** how many values of `i` does each stop give for 9 letters?

In [ ]:
letters = "ἄνδρα μοι"
chunk_size = 4
print("length:", len(letters))
print("with - 1   :", list(range(0, len(letters) - chunk_size - 1, chunk_size)))
print("without - 1:", list(range(0, len(letters) - chunk_size, chunk_size)))
for i in range(0, len(letters) - chunk_size, chunk_size):
    print(i, letters[i : i + chunk_size], "→", letters[i + 1 : i + chunk_size + 1])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

With `- 1` the stop gives `[0]`; without it, `[0, 4]`. And the second chunk (`α μο` → ` μοι`) fits exactly: its targets end at position 8, the last letter. On Book 1 (19,701 characters) both stops happen to give the same 820 chunks, so the trained model comes out the same either way.

</details>

## 5. Recap

- The model practises on flashcards: some text, and the character that comes next.
- The text itself provides the answers, so this is self-supervised learning.
- One chunk of 24 characters, plus a copy shifted by one, gives 24 flashcards at once.

**You can now:**

- cut the inputs and the targets for any start `i`
- read two aligned rows as flashcards, column by column
- say why the loop stops at `len(data) - chunk_size`

**Next:** [02_05: Token embeddings](02_05_token_embeddings.ipynb): turning each token number into a list of numbers that can carry meaning.